In [1]:
# Useful imports
import os
from pathlib import Path
import tempfile
import hydra
import sys

### User Configuration Section

In [2]:
# Pick the parent directory; the next cell walks it to collect every .nuboard file inside.
# Swap 'closed_loop_reactive_agents' <-> 'closed_loop_nonreactive_agents' to view the other mode.
# Or point at a single run directory (e.g. .../grpoCL_clR_mini_20260615_212911) to view just one.
RESULT_FOLDER = "/home/noopur/Documents/nuplan/exp/exp/simulation/closed_loop_reactive_agents"

env_variables = {
    "NUPLAN_DEVKIT_ROOT": "/home/noopur/Documents/nuplan-devkit",
    "NUPLAN_DATA_ROOT":   "/home/noopur/Documents/nuplan/dataset",
    "NUPLAN_MAPS_ROOT":   "/home/noopur/Documents/nuplan/dataset/maps",
    "NUPLAN_DB_FILES":    "/home/noopur/Documents/nuplan/dataset/nuplan-v1.1/splits/mini",
    "NUPLAN_EXP_ROOT":    "/home/noopur/Documents/nuplan/exp",
    "NUPLAN_SIMULATION_ALLOW_ANY_BUILDER": "1",
}
for k, v in env_variables.items():
    os.environ[k] = v

# Make the devkit source tree importable AHEAD of any pip-installed nuplan in site-packages.
# The pip-installed copy ships no config YAMLs, so Hydra fails to resolve `pkg://nuplan...` without this.
for p in [env_variables["NUPLAN_DEVKIT_ROOT"], os.getcwd()]:
    while p in sys.path:
        sys.path.remove(p)
    sys.path.insert(0, p)

# Evict any stale nuplan modules cached from a previous cell run, so the next import
# picks up the devkit source tree (not the site-packages copy from earlier).
for mod in [m for m in list(sys.modules) if m == 'nuplan' or m.startswith('nuplan.')]:
    del sys.modules[mod]

# Sanity check: confirm we resolve to the devkit, and the config YAMLs exist.
import nuplan
_resolved = os.path.dirname(nuplan.__file__)
print('nuplan resolved to:', _resolved)
assert _resolved.startswith(env_variables['NUPLAN_DEVKIT_ROOT']), (
    f'nuplan is being imported from {_resolved} (site-packages, no YAMLs). '
    f'Restart the kernel and re-run all cells from the top.'
)
for rel in ['planning/script/config/common/default_common.yaml',
            'planning/script/config/common/splitter/nuplan.yaml',
            'planning/script/config/common/scenario_builder/nuplan_mini.yaml']:
    full = os.path.join(_resolved, rel)
    print(('OK  ' if os.path.isfile(full) else 'MISS'), full)

# Location of path with all nuBoard configs (relative to this notebook's directory)
CONFIG_PATH = '../nuplan-devkit/nuplan/planning/script/config/nuboard'

nuplan resolved to: /home/noopur/Documents/nuplan-devkit/nuplan
OK   /home/noopur/Documents/nuplan-devkit/nuplan/planning/script/config/common/default_common.yaml
OK   /home/noopur/Documents/nuplan-devkit/nuplan/planning/script/config/common/splitter/nuplan.yaml
OK   /home/noopur/Documents/nuplan-devkit/nuplan/planning/script/config/common/scenario_builder/nuplan_mini.yaml


In [3]:
CONFIG_NAME = 'default_nuboard'

# Initialize configuration management system
hydra.core.global_hydra.GlobalHydra.instance().clear()  # reinitialize hydra if already initialized
hydra.initialize(config_path=CONFIG_PATH)

ml_planner_simulation_folder = RESULT_FOLDER
ml_planner_simulation_folder = [dp for dp, _, fn in os.walk(ml_planner_simulation_folder) if True in ['.nuboard' in x for x in fn]]

# Compose the configuration
cfg = hydra.compose(config_name=CONFIG_NAME, overrides=[
    'scenario_builder=nuplan_mini',  # mini split, matches all closed-loop runs in this repo
    f'simulation_path={ml_planner_simulation_folder}',  # collected .nuboard files
    'hydra.searchpath=[pkg://diffusion_planner.config.scenario_filter, pkg://diffusion_planner.config, pkg://nuplan.planning.script.config.common, pkg://nuplan.planning.script.experiments]',
    'port_number=6599'
])
print(f'Loaded {len(ml_planner_simulation_folder)} .nuboard run(s):')
for r in ml_planner_simulation_folder:
    print('  ', r)

/tmp/ipykernel_28894/2909217188.py:5: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  hydra.initialize(config_path=CONFIG_PATH)
/home/noopur/anaconda3/envs/diffusion_planner/lib/python3.9/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'default_nuboard': Defaults list is missing `_self_`. See https://hydra.cc/docs/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)


Loaded 11 .nuboard run(s):
   /home/noopur/Documents/nuplan/exp/exp/simulation/closed_loop_reactive_agents/sft_clR_mini_20260615_193618
   /home/noopur/Documents/nuplan/exp/exp/simulation/closed_loop_reactive_agents/grpoCL_clR_mini_20260615_175507
   /home/noopur/Documents/nuplan/exp/exp/simulation/closed_loop_reactive_agents/grpoCL_clR_mini_20260615_212911
   /home/noopur/Documents/nuplan/exp/exp/simulation/closed_loop_reactive_agents/sft_clR_mini_20260615_175507
   /home/noopur/Documents/nuplan/exp/exp/simulation/closed_loop_reactive_agents/sft_clR_mini_20260615_170307
   /home/noopur/Documents/nuplan/exp/exp/simulation/closed_loop_reactive_agents/sft_clR_mini_20260615_212911
   /home/noopur/Documents/nuplan/exp/exp/simulation/closed_loop_reactive_agents/grpo_clR_mini_20260615_175507
   /home/noopur/Documents/nuplan/exp/exp/simulation/closed_loop_reactive_agents/grpo_clR_mini_20260615_193618
   /home/noopur/Documents/nuplan/exp/exp/simulation/closed_loop_reactive_agents/grpo_clR_mini

In [ ]:
from nuplan.planning.script.run_nuboard import main as main_nuboard

# Run nuBoard
main_nuboard(cfg)

/home/noopur/Documents/nuplan-devkit/nuplan/planning/script/run_nuboard.py:67: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  @hydra.main(config_path=CONFIG_PATH, config_name=CONFIG_NAME)
INFO:nuplan.planning.script.builders.scenario_building_builder:Building AbstractScenarioBuilder...
INFO:nuplan.planning.script.builders.scenario_building_builder:Building AbstractScenarioBuilder...DONE!
INFO:nuplan.planning.nuboard.nuboard:Opening Bokeh application on http://localhost:6599/
INFO:nuplan.planning.nuboard.nuboard:Async rendering is set to: True
INFO:bokeh.server.server:Starting Bokeh server version 2.4.3 (running on Tornado 6.5.5)
INFO:bokeh.server.tornado:User authentication hooks NOT provided (default user enabled)
INFO:nuplan.planning.nuboard.base.simulation_tile:Minimum frame time=0.017 s
INFO:nuplan.planning.nuboard.tabs.scenario_tab:Rending scenario plot takes 0.0014 seconds.
I